# Topic 11 — XGBoost (Extreme Gradient Boosting)

> **Goal of this notebook:** understand what makes XGBoost the workhorse of tabular ML competitions — a regularised, second-order, highly optimised version of gradient boosting — and use its key features: early stopping and feature importance.

**Contents**
1. Concept & intuition
2. The mathematics (regularised objective, second-order approximation)
3. What XGBoost adds over plain gradient boosting
4. The dataset: Breast Cancer
5. Training with early stopping
6. Evaluation & feature importance
7. Comparison vs scikit-learn Gradient Boosting
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

XGBoost is gradient boosting (Topic 10) — same additive, residual-correcting idea — but engineered for **speed, accuracy, and regularisation**. It became dominant because it:
- adds **L1/L2 regularisation** on leaf weights to fight overfitting,
- uses a **second-order (Newton) approximation** of the loss for better splits,
- is **sparsity-aware** (handles missing values natively),
- supports **early stopping**, column subsampling, and parallelised tree construction.

Same core algorithm, far better defaults and engineering.

## 2. The Mathematics

XGBoost minimises a **regularised objective** — loss plus a penalty on tree complexity:

$$\mathcal{L} = \sum_i L(y_i, \hat{y}_i) + \sum_k \Omega(f_k), \qquad \Omega(f) = \gamma T + \tfrac{1}{2}\lambda \sum_j w_j^2$$

where $T$ is the number of leaves and $w_j$ the leaf weights. At each boosting step it expands the loss to **second order** (Taylor) using the gradient $g_i$ and Hessian $h_i$:

$$\mathcal{L}^{(t)} \approx \sum_i \big[ g_i f_t(x_i) + \tfrac{1}{2} h_i f_t(x_i)^2 \big] + \Omega(f_t)$$

This yields a closed-form **optimal leaf weight** and a **split-gain** formula that explicitly accounts for regularisation — giving better, more conservative splits than first-order gradient boosting.

## 3. What XGBoost Adds Over Plain Gradient Boosting

| Feature | Benefit |
|---------|---------|
| `reg_alpha` (L1), `reg_lambda` (L2) | regularises leaf weights → less overfitting |
| 2nd-order (Hessian) splits | more accurate, faster convergence |
| `subsample`, `colsample_bytree` | row/column sampling → decorrelation |
| early stopping | stops when validation stops improving |
| native missing-value handling | no imputation needed |
| parallel & cache-aware | much faster training |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

print('xgboost version:', xgb.__version__)

## 4. The Dataset: Breast Cancer

We carve out a validation set from the training data so XGBoost can use **early stopping** without touching the test set.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)
print('Train:', X_train.shape, ' Val:', X_val.shape, ' Test:', X_test.shape)

## 5. Training with Early Stopping

We allow up to 500 trees but stop once the validation log-loss hasn't improved for 20 rounds.

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1, max_depth=3,
    subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, reg_alpha=0.0,
    eval_metric='logloss', early_stopping_rounds=20,
    random_state=42)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print('Best iteration (trees actually used):', model.best_iteration)
print('Test accuracy:', round(accuracy_score(y_test, model.predict(X_test)), 4))

In [ ]:
# Validation log-loss curve over boosting rounds
results = model.evals_result()
ll = results['validation_0']['logloss']
plt.plot(range(1, len(ll) + 1), ll)
plt.axvline(model.best_iteration + 1, color='r', ls='--', label='best iteration')
plt.xlabel('boosting round'); plt.ylabel('validation log-loss')
plt.title('Early stopping picks the best number of trees'); plt.legend(); plt.show()

## 6. Evaluation & Feature Importance

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=data.target_names))

imp = model.feature_importances_
order = np.argsort(imp)[-10:]
plt.barh([data.feature_names[i] for i in order], imp[order])
plt.xlabel('importance (gain)'); plt.title('Top 10 features (XGBoost)'); plt.show()

## 7. Comparison vs scikit-learn Gradient Boosting

In [ ]:
gbm = GradientBoostingClassifier(n_estimators=model.best_iteration + 1,
                                learning_rate=0.1, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

print('sklearn GradientBoosting test accuracy:', round(gbm.score(X_test, y_test), 4))
print('XGBoost                test accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 8. Pros, Cons & When to Use

**Pros**
- Best-in-class accuracy on tabular data; the default for competitions.
- Built-in regularisation, early stopping, and missing-value handling.
- Fast and parallelised; rich tuning controls.

**Cons**
- Many hyperparameters → tuning takes effort.
- Can overfit if under-regularised; less interpretable.
- Overkill for tiny or simple datasets.

**When to use**
- Serious tabular classification/regression where accuracy is the goal.
- Whenever plain gradient boosting is a candidate, XGBoost (or LightGBM) is usually the stronger choice.

## 9. Summary

- XGBoost is gradient boosting with a **regularised objective** and a **second-order** split criterion, plus heavy engineering.
- **Early stopping** chose the number of trees automatically from a validation curve.
- It performed comparably to scikit-learn's gradient boosting here (on small data they are close; XGBoost's edge grows on larger, noisier datasets) while being faster and more robust.
- Regularisation (`reg_lambda`/`reg_alpha`) and subsampling are its main overfitting defences.